# Heart Disease Prediction with Logistic Regression
AI 240 - Homework 2 Lab

## 1. Data Exploration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)

sns.set_style('whitegrid')
df = pd.read_csv('heart.csv')
df.head()

In [ ]:
print(df.shape)
df.info()

In [ ]:
df.describe().T

In [ ]:
print(df.isnull().sum())
print('Duplicate rows:', df.duplicated().sum())

Target variable: `target` (1 = heart disease, 0 = no heart disease). The other 13 columns are features.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(x='target', data=df, ax=ax)
ax.set_title('Target Distribution')
ax.set_xlabel('target (0 = no disease, 1 = disease)')
plt.show()
df['target'].value_counts(normalize=True)

In [ ]:
num_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
df[num_cols].hist(bins=20, figsize=(14, 7), layout=(2, 3))
plt.suptitle('Numerical Feature Distributions')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, col in zip(axes, num_cols):
    sns.boxplot(x='target', y=col, data=df, ax=ax)
    ax.set_title(f'{col} by target')
plt.tight_layout()
plt.show()

In [ ]:
cat_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.ravel(), cat_cols):
    sns.countplot(x=col, hue='target', data=df, ax=ax)
    ax.set_title(f'{col} vs target')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(11, 8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.show()
df.corr()['target'].drop('target').sort_values()

## 2. Data Preprocessing

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
print(df.shape)
print(df['target'].value_counts())

The raw file contains many exact duplicate rows. They are removed so identical patients cannot appear in both the train and test sets.

In [ ]:
df_model = pd.get_dummies(df, columns=['cp', 'restecg', 'slope', 'thal'], drop_first=True)
X = df_model.drop('target', axis=1)
y = df_model['target']
X.shape

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scale_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
scaler = StandardScaler()
X_train_s = X_train.copy()
X_test_s = X_test.copy()
X_train_s[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test_s[scale_cols] = scaler.transform(X_test[scale_cols])
print(X_train_s.shape, X_test_s.shape)

## 3. Modeling

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_s, y_train)
y_pred = model.predict(X_test_s)
y_prob = model.predict_proba(X_test_s)[:, 1]

In [ ]:
coef = pd.Series(model.coef_[0], index=X.columns).sort_values()
coef.plot(kind='barh', figsize=(8, 7), title='Logistic Regression Coefficients')
plt.show()

## 4. Evaluation

In [ ]:
print('Accuracy :', round(accuracy_score(y_test, y_pred), 4))
print('Precision:', round(precision_score(y_test, y_pred), 4))
print('Recall   :', round(recall_score(y_test, y_pred), 4))
print('F1 score :', round(f1_score(y_test, y_pred), 4))
print('ROC AUC  :', round(roc_auc_score(y_test, y_prob), 4))
print()
print(classification_report(y_test, y_pred, target_names=['No disease', 'Disease']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                       display_labels=['No disease', 'Disease']).plot(ax=axes[0], cmap='Blues')
axes[0].set_title('Confusion Matrix')
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, label=f'AUC = {roc_auc_score(y_test, y_prob):.3f}')
axes[1].plot([0, 1], [0, 1], 'k--')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
X_all = X.copy()
X_all[scale_cols] = StandardScaler().fit_transform(X_all[scale_cols])
cv = cross_val_score(LogisticRegression(max_iter=1000), X_all, y, cv=5, scoring='accuracy')
print('5-fold CV accuracy:', cv.round(3))
print('Mean: %.4f  Std: %.4f' % (cv.mean(), cv.std()))

In [ ]:
grid = GridSearchCV(LogisticRegression(max_iter=1000),
                    {'C': [0.01, 0.1, 1, 10, 100]},
                    cv=5, scoring='accuracy')
grid.fit(X_train_s, y_train)
print('Best params:', grid.best_params_)
best_pred = grid.best_estimator_.predict(X_test_s)
print('Tuned test accuracy:', round(accuracy_score(y_test, best_pred), 4))
print('Tuned test recall  :', round(recall_score(y_test, best_pred), 4))

## Conclusion
The logistic regression model predicts heart disease on unseen patients with good accuracy and ROC AUC. Recall matters most in this setting because a missed case (false negative) is more costly than a false alarm. Cross-validation shows the results are stable across splits, and tuning `C` gives a small improvement on the test set.